## Traditional Attempt at classification (Random Forest & XGBoost)

This is a first attempt at classification of the combined dataset using Random Forest.

Reading in the data

In [13]:
#Bad things happen when you uncomment these lines
# Beware
#df = pd.read_parquet('../data/MERGED_LCS.parquet')
#df

Reducing the precision of the timestamps and flux values so my kernel stops crashing

In [3]:
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import pandas as pd

In [4]:

infile = "../data/MERGED_LCS.parquet"
outfile = "../data/MERGED_LCS_reduced.parquet"

def round_array(x, ndigits=5):
    if x is None:
        return None
    arr = np.asarray(x, dtype=np.float32)
    return np.round(arr, ndigits).tolist()

pf = pq.ParquetFile(infile)
writer = None

for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    # Convert directly from Arrow to a normal Python dict
    data = batch.to_pydict()

    # Build the dataframe explicitly, including KIC
    df = pd.DataFrame({
        "time": data["time"],
        "flux": data["flux"],
        "Class": data["Class"],
        "KIC": data["KIC"],
    })

    df["time"] = df["time"].apply(round_array)
    df["flux"] = df["flux"].apply(round_array)

    # Save this batch
    table = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(outfile, table.schema, compression="zstd")

    writer.write_table(table)

if writer is not None:
    writer.close()

In [5]:
batch = next(pf.iter_batches(batch_size=1, columns=["time", "flux", "Class", "KIC"]))
df = batch.to_pandas(ignore_metadata=True)
print(df.columns)
print(df.head())

Index(['time', 'flux', 'Class', 'KIC'], dtype='object')
                                                time  \
0  [131.51271468758932, 131.5331494016791, 131.55...   

                                                flux      Class     KIC  
0  [1.0392150925008297, 1.0383358659303283, 1.038...  CONFIRMED  757450  


In [5]:
df = pd.read_parquet("../data/MERGED_LCS_reduced.parquet")

In [6]:
df

,time,flux,Class,KIC
0,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0392199754714966, 1.0383399724960327, 1.038...",CONFIRMED,757450
1,"[352.39654541015625, 352.43743896484375, 352.4...","[1.019569993019104, 1.018839955329895, 1.01899...",FALSE POSITIVE,892772
2,"[120.539306640625, 120.55975341796875, 120.580...","[1.0488500595092773, 1.0523099899291992, 1.051...",CANDIDATE,1025986
3,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0509400367736816, 1.0503300428390503, 1.050...",FALSE POSITIVE,1026032
4,"[120.53929901123047, 120.55973815917969, 120.5...","[1.059939980506897, 1.0591700077056885, 1.0589...",CONFIRMED,1026957
...,...,...,...,...
18877,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9961699843406677, 0.9965299963951111, 0.996...",VARIABLE STAR,202140012
18878,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9897400140762329, 0.9907699823379517, 0.991...",VARIABLE STAR,202140013
18879,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0229099988937378, 1.014359951019287, 1.0042...",FALSE POSITIVE,202140059
18880,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0080900192260742, 1.0085899829864502, 1.009...",FALSE POSITIVE,202140094


## Preprocessing

In [16]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import median_filter

def sigma_clip_mad(t, f, sigma=5.0):
    """
    Robust outlier removal using MAD.
    Returns filtered time and flux arrays.
    """
    if len(f) == 0:
        return t, f

    med = np.median(f)
    mad = np.median(np.abs(f - med))

    if mad == 0:
        return t, f

    robust_sigma = 1.4826 * mad
    keep = np.abs(f - med) <= sigma * robust_sigma
    return t[keep], f[keep]

def normalize_by_median(f):
    """
    Normalize flux by its median.
    """
    med = np.median(f)
    if med == 0:
        return f
    return f / med

def detrend_with_median_filter(t, f, window_frac=0.1):
    """
    Detrend by interpolating onto a uniform grid and subtracting a median-filter trend.
    Works best after normalization.
    """
    n = len(f)
    if n < 10:
        return f

    # Interpolate onto a uniform grid
    grid = np.linspace(t.min(), t.max(), n, dtype=np.float32)
    f_interp = np.interp(grid, t, f).astype(np.float32)

    # Median filter window size
    win = max(5, int(n * window_frac))
    if win >= n:
        win = n - 1
    if win % 2 == 0:
        win -= 1
    if win < 5:
        return f

    trend = median_filter(f_interp, size=win, mode="nearest")
    trend_at_t = np.interp(t, grid, trend).astype(np.float32)

    # Avoid divide-by-zero
    trend_at_t = np.where(np.abs(trend_at_t) < 1e-8, 1.0, trend_at_t)

    # Since normalized flux is usually around 1, division is okay here
    return f / trend_at_t


def preprocess_lightcurve(time, flux, sigma=5.0, detrend=True, n_points=200):
    t = np.asarray(time, dtype=np.float32)
    f = np.asarray(flux, dtype=np.float32)

    mask = np.isfinite(t) & np.isfinite(f)
    t = t[mask]
    f = f[mask]

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    order = np.argsort(t)
    t = t[order]
    f = f[order]

    f = normalize_by_median(f)
    t, f = sigma_clip_mad(t, f, sigma=sigma)

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    if detrend and len(t) >= 10:
        f = detrend_with_median_filter(t, f)

    # If detrending or clipping produced bad values
    mask = np.isfinite(t) & np.isfinite(f)
    t = t[mask]
    f = f[mask]

    if len(t) < 2:
        return np.zeros(n_points, dtype=np.float32)

    t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    t_new = np.linspace(0, 1, n_points, dtype=np.float32)
    f_new = np.interp(t_new, t, f).astype(np.float32)

    return f_new

In [17]:
infile = "../data/MERGED_LCS.parquet"
pf = pq.ParquetFile(infile)

cleaned_rows = []

for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    df = batch.to_pandas(ignore_metadata=True)

    for t, f, label, kic in zip(df["time"], df["flux"], df["Class"], df["KIC"]):
        vec = preprocess_lightcurve(t, f)

        cleaned_rows.append({
            "features": vec,
            "Class": label,
            "KIC": kic
        })

cleaned_df = pd.DataFrame(cleaned_rows)

In [18]:
cleaned_df

,features,Class,KIC
0,"[1.0, 1.0142351, 0.9917163, 1.0041176, 1.00521...",CONFIRMED,757450
1,"[1.0, 0.99980605, 0.999324, 1.0000539, 0.99972...",FALSE POSITIVE,892772
2,"[1.0, 0.993915, 0.9882146, 0.99423313, 1.01133...",CANDIDATE,1025986
3,"[1.0, 0.99872774, 1.0031949, 1.0013711, 1.0008...",FALSE POSITIVE,1026032
4,"[1.0, 1.0000197, 0.99358433, 1.0007925, 0.9877...",CONFIRMED,1026957
...,...,...,...
18877,"[1.0, 1.0007939, 1.000708, 1.001362, 1.0005054...",VARIABLE STAR,202140012
18878,"[1.0, 0.99774355, 1.0011828, 1.0009707, 0.9910...",VARIABLE STAR,202140013
18879,"[1.0, 1.015791, 1.006078, 1.0207278, 1.022889,...",FALSE POSITIVE,202140059
18880,"[1.0, 0.99416035, 0.9960629, 0.9960192, 0.9942...",FALSE POSITIVE,202140094


# Random Forest Model

In [19]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Build X and y
X = np.vstack(cleaned_df["features"].values)
y = cleaned_df["Class"].values
kic = cleaned_df["KIC"].values   # keep for reference, not training

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test, kic_train, kic_test = train_test_split(
    X, y_enc, kic,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5440826052422557

Classification report:
                       precision    recall  f1-score   support

            CANDIDATE       0.00      0.00      0.00       328
            CONFIRMED       0.27      0.01      0.01       395
ECLIPSING BINARY STAR       0.64      0.36      0.46       642
       FALSE POSITIVE       0.52      0.92      0.67      1816
        VARIABLE STAR       0.71      0.27      0.39       596

             accuracy                           0.54      3777
            macro avg       0.43      0.31      0.31      3777
         weighted avg       0.50      0.54      0.46      3777


Confusion matrix:
[[   0    6   10  312    0]
 [   0    3    4  388    0]
 [   0    0  229  411    2]
 [   1    2   86 1664   63]
 [   0    0   30  407  159]]
